<a href="https://colab.research.google.com/github/praneetha-glitch/AI-Lab/blob/main/WEEK_9_FOL_Resolution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
from itertools import combinations


def is_variable(term):
    return term.islower() and "(" not in term


def parse_literal(lit):
    neg = lit.startswith("~")
    clean = lit[1:] if neg else lit
    pred, rest = clean.split("(")
    args = [arg.strip() for arg in rest[:-1].split(",")]
    return neg, pred.strip(), args


def format_literal(neg, pred, args):
    prefix = "~" if neg else ""
    return f"{prefix}{pred}({', '.join(args)})"


def negate_literal(lit):
    return lit[1:] if lit.startswith("~") else "~" + lit


def apply_theta_term(term, theta):
    while term in theta:
        term = theta[term]
    return term


def substitute(clause, theta):
    new_clause = []
    for lit in clause:
        neg, pred, args = parse_literal(lit)
        new_args = [apply_theta_term(arg, theta) for arg in args]
        new_clause.append(format_literal(neg, pred, new_args))
    return sorted(list(set(new_clause)))


def normalize_variables(clause):
    mapping = {}
    new_clause = []
    idx = 1
    for lit in sorted(clause):
        neg, pred, args = parse_literal(lit)
        new_args = []
        for arg in args:
            if is_variable(arg):
                if arg not in mapping:
                    mapping[arg] = f"x{idx}"
                    idx += 1
                new_args.append(mapping[arg])
            else:
                new_args.append(arg)
        new_clause.append(format_literal(neg, pred, new_args))
    return sorted(list(set(new_clause)))


def standardize_apart(ci, cj):
    ci_norm = normalize_variables(ci)
    mapping = {}
    cj_renamed = []
    idx = 1
    for lit in cj:
        neg, pred, args = parse_literal(lit)
        new_args = []
        for arg in args:
            if is_variable(arg):
                if arg not in mapping:
                    mapping[arg] = f"y{idx}"
                    idx += 1
                new_args.append(mapping[arg])
            else:
                new_args.append(arg)
        cj_renamed.append(format_literal(neg, pred, new_args))
    return ci_norm, cj_renamed


def unify_args(args1, args2):
    if len(args1) != len(args2):
        return None
    theta = {}
    for a, b in zip(args1, args2):
        a = apply_theta_term(a, theta)
        b = apply_theta_term(b, theta)
        if a == b:
            continue
        elif is_variable(a):
            theta[a] = b
        elif is_variable(b):
            theta[b] = a
        else:
            return None
    return theta


def resolve(ci, cj):
    ci_std, cj_std = standardize_apart(ci, cj)
    resolvents = []

    for i, lit1 in enumerate(ci_std):
        neg1, pred1, args1 = parse_literal(lit1)
        for j, lit2 in enumerate(cj_std):
            neg2, pred2, args2 = parse_literal(lit2)

            if pred1 == pred2 and neg1 != neg2:
                theta = unify_args(args1, args2)
                if theta is not None:
                    rem1 = [l for k, l in enumerate(ci_std) if k != i]
                    rem2 = [l for k, l in enumerate(cj_std) if k != j]
                    combined = normalize_variables(
                        substitute(rem1 + rem2, theta)
                    )
                    resolvents.append(combined)
    return resolvents


def fol_resolution(kb, query):
    clauses = [normalize_variables(c) for c in kb]
    negated_goal = normalize_variables(
        [negate_literal(q.strip()) for q in query.split("|")]
    )
    clauses.append(negated_goal)

    unique_clauses = []
    for c in clauses:
        if c not in unique_clauses:
            unique_clauses.append(c)
    clauses = unique_clauses

    print("\nInitial CNF Clauses (KB + Negated Goal):")
    for idx, c in enumerate(clauses, 1):
        print(f"  {idx}. {c}")
    print("-" * 55)

    parents = {}

    while True:
        new_clauses = []
        for ci, cj in combinations(clauses, 2):
            for res in resolve(ci, cj):
                key = tuple(res)
                if res not in clauses and res not in new_clauses:
                    parents[key] = (ci, cj)
                    if len(res) == 0:
                        steps = []
                        stack = [key]
                        visited = set()
                        while stack:
                            curr = stack.pop()
                            if curr in parents and curr not in visited:
                                visited.add(curr)
                                p1, p2 = parents[curr]
                                steps.append((p1, p2, list(curr)))
                                stack.append(tuple(p1))
                                stack.append(tuple(p2))
                        for p1, p2, r in reversed(steps):
                            print(
                                f"Resolve {p1} & {p2}  ==>  "
                                f"{r if r else 'EMPTY CLAUSE []'}"
                            )
                        print("-" * 55)
                        print(f"RESULT: Query '{query}' CAN BE PROVED.")
                        return True
                    new_clauses.append(res)

        if not new_clauses:
            print("No empty clause could be derived.")
            print("-" * 55)
            print(
                f"RESULT: Query '{query}' CANNOT BE PROVED from the given KB."
            )
            return False

        clauses.extend(new_clauses)


n = int(input("Enter number of CNF clauses in Knowledge Base: "))
print("Enter each clause using '|' for OR and '~' for NOT:")
print("Example: ~Food(x) | Likes(John, x)")

knowledge_base = []
for i in range(n):
    raw = input(f"Clause {i + 1}: ")
    clause = [lit.strip() for lit in raw.split("|") if lit.strip()]
    knowledge_base.append(clause)

query_input = input("\nEnter the Goal/Query to prove: ").strip()
fol_resolution(knowledge_base, query_input)


Enter number of CNF clauses in Knowledge Base: 8
Enter each clause using '|' for OR and '~' for NOT:
Example: ~Food(x) | Likes(John, x)
Clause 1: ~Food(x) | Likes(John, x)
Clause 2: Food(Apples)
Clause 3: Food(Chicken)
Clause 4: ~Eats(y, z) | Killed(y) | Food(z)
Clause 5: Eats(Bill, Peanuts)
Clause 6: Alive(Bill)
Clause 7: ~Eats(Bill, w) | Eats(Sue, w)
Clause 8: ~Alive(u) | ~Killed(u)

Enter the Goal/Query to prove: Likes(John, Peanuts)

Initial CNF Clauses (KB + Negated Goal):
  1. ['Likes(John, x1)', '~Food(x1)']
  2. ['Food(Apples)']
  3. ['Food(Chicken)']
  4. ['Food(x1)', 'Killed(x2)', '~Eats(x2, x1)']
  5. ['Eats(Bill, Peanuts)']
  6. ['Alive(Bill)']
  7. ['Eats(Sue, x1)', '~Eats(Bill, x1)']
  8. ['~Alive(x1)', '~Killed(x1)']
  9. ['~Likes(John, Peanuts)']
-------------------------------------------------------
Resolve ['Likes(John, x1)', '~Food(x1)'] & ['~Likes(John, Peanuts)']  ==>  ['~Food(Peanuts)']
Resolve ['Food(x1)', 'Killed(x2)', '~Eats(x2, x1)'] & ['Eats(Bill, Peanuts)']

True